Adding more features

In [1]:
import os
import glob
import pandas as pd
import json
import numpy as np
from sklearn.preprocessing import StandardScaler

In [3]:
print("Loading cleaned training data...")
df_clean = pd.read_parquet('../data/cleaned/train_hospital_A_clean.parquet')

# Drop ICULOS as ICU_Hour is superior
if 'ICULOS' in df_clean.columns:
    df_clean = df_clean.drop(columns=['ICULOS'])

print("Engineering the Top-Tier Dynamic Features...")

# 1. System Overload Score (Ranked #2 Overall)
df_clean['System_Overload_Score'] = (
    (df_clean['HR'] > 100).astype(int) +
    (df_clean['SBP'] < 90).astype(int) +
    (df_clean['Resp'] > 22).astype(int) +
    ((df_clean['Temp'] > 38) | (df_clean['Temp'] < 36)).astype(int)
)

# 2. Age-Adjusted Frailty Index (Ranked #5)
df_clean['Age_Frailty_Index'] = (df_clean['Age'] / 100) * df_clean['System_Overload_Score']

# 3. Oxygen Demand Dynamics (FiO2 - Ranked #4 and #6)
df_clean['FiO2_4hr_mean'] = df_clean.groupby('Patient_ID')['FiO2'].transform(lambda x: x.rolling(window=4, min_periods=1).mean())
df_clean['FiO2_4hr_std'] = df_clean.groupby('Patient_ID')['FiO2'].transform(lambda x: x.rolling(window=4, min_periods=2).std())

# 4. Temperature Trend (Ranked #10)
df_clean['Temp_4hr_mean'] = df_clean.groupby('Patient_ID')['Temp'].transform(lambda x: x.rolling(window=4, min_periods=1).mean())

# 5. Admission Delta for Oxygen Saturation (Ranked #13)
o2sat_baseline = df_clean.groupby('Patient_ID')['O2Sat'].transform('first')
df_clean['O2Sat_Admission_Delta'] = df_clean['O2Sat'] - o2sat_baseline

# 6. Systolic Blood Pressure Volatility (Ranked #14 & #16)
df_clean['SBP_4hr_mean'] = df_clean.groupby('Patient_ID')['SBP'].transform(lambda x: x.rolling(window=4, min_periods=1).mean())
df_clean['SBP_4hr_std'] = df_clean.groupby('Patient_ID')['SBP'].transform(lambda x: x.rolling(window=4, min_periods=2).std())
df_clean['SBP_Volatility'] = df_clean['SBP_4hr_std'] / (df_clean['SBP_4hr_mean'] + 1e-5)

# Notice: We are completely skipping the .fillna(0) step! We want those NaNs to stay.

print("Saving lean, optimized dataset...")
out_path = '../data/final/train_hospital_A_optimized.parquet'
df_clean.to_parquet(out_path, index=False)

print(f"Feature engineering complete. Dataset shape: {df_clean.shape}")

Loading cleaned training data...
Engineering the Top-Tier Dynamic Features...
Saving lean, optimized dataset...
Feature engineering complete. Dataset shape: (783819, 50)


Because I am using a XGBoost model I will skip the NORMALIZATION.